In [3]:
# Run this in a cell before exporting
%pip install onnx onnxsim onnxruntime-gpu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
from ultralytics import YOLO
import yaml
import torch
torch.cuda.empty_cache()


DATASET_PATH = "rebalanced_dataset"

In [5]:
def create_dataset_yaml(dataset_path="augmentation_dataset", num_classes=1, class_names=None):
    """Create dataset YAML configuration with proper formatting"""
    if class_names is None:
        class_names = {0: 'pest'}
    
    yaml_content = {
        'path': os.path.abspath(dataset_path),
        'train': 'train/images',
        'val': 'valid/images', 
        'test': 'test/images',
        'nc': num_classes,
        'names': class_names
    }
    
    # Save YAML
    with open("data.yaml", "w") as f:
        yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)
    
    print("✅ data.yaml created!")
    print(f"Dataset path: {os.path.abspath(dataset_path)}")
    return "data.yaml"

# Create dataset configuration
yaml_path = create_dataset_yaml(DATASET_PATH, num_classes=2, class_names={0: 'raccoon', 1: 'rodent'}) # changed to try two-class system for better precision

✅ data.yaml created!
Dataset path: c:\Users\plebj\Desktop\School\CS5814\rebalanced_dataset


In [6]:
def verify_dataset_structure(dataset_path):
    """Verify dataset has correct structure"""
    required_dirs = [
        "train/images", "train/labels",
        "valid/images", "valid/labels", 
        "test/images", "test/labels"
    ]
    
    print("🔍 Verifying dataset structure...")
    for dir_path in required_dirs:
        full_path = os.path.join(dataset_path, dir_path)
        if not os.path.exists(full_path):
            print(f"❌ Missing: {full_path}")
            return False
        # Check if directory has files
        files = os.listdir(full_path)
        if not files:
            print(f"⚠️  Empty directory: {full_path}")
        else:
            print(f"✅ {dir_path}: {len(files)} files")
    
    return True

# Verify dataset
dataset_valid = verify_dataset_structure(DATASET_PATH)
if not dataset_valid:
    print("❌ Dataset structure invalid! Fix before training.")

🔍 Verifying dataset structure...
✅ train/images: 1246 files
✅ train/labels: 1246 files
✅ valid/images: 356 files
✅ valid/labels: 356 files
✅ test/images: 178 files
✅ test/labels: 178 files


In [7]:
def train_model(yaml_path="data.yaml"):
    """Main training function with enhanced configuration"""

    # Load YOLO11 model
    print("🚀 Loading YOLO11n model...")
    model = YOLO("yolo11n.pt")
    
    # Training configuration
    results = model.train(
        data=yaml_path,
        epochs=100,
        imgsz=640,
        batch=16,
        device=0,
        single_cls=False,       # Now we are doing multi-class


        # === CONSERVATIVE BUILT-IN AUGMENTATIONS ===
        hsv_h=0.015,    # Minimal hue variation
        hsv_s=0.6,      # Moderate saturation  
        hsv_v=0.4,      # Conservative brightness
        degrees=3.0,     # Small rotations only
        translate=0.1,   # Small translations
        scale=0.2,       # Conservative scaling
        fliplr=0.5,      # Only horizontal flips (realistic)  

        # === DISABLE COMPLEX AUGMENTATIONS ===
        mosaic=0.0,      # No mosaic (unrealistic contexts)
        mixup=0.0,       # No mixup (unrealistic blends) 
        copy_paste=0.0,  # No copy-paste (artifacts)

        # === REGULARIZATION ===
        weight_decay=0.0005,  # Prevent overfitting
        patience=15,          # Early stopping patience
        lr0=0.01,             # Learning rate
        warmup_epochs=3,      # Learning rate warmup
        
        # === LOSS WEIGHTS ===
        box=7.5,        # Box loss gain
        cls=1.0,        # Increased class loss for better class distinction
        dfl=1.5,        # Distribution focal loss

        verbose=True,          # Print results per epoch
    )
    
    return model, results

# Train the model (this will take ~25 minutes)
model, results = train_model(yaml_path)

🚀 Loading YOLO11n model...
New https://pypi.org/project/ultralytics/8.3.227 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.204  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=3.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.6, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=0.0, multi_scale=False, name=train, nbs=64, nms

In [8]:
def simple_evaluate_model(model, yaml_path="data.yaml"):
    """Simple baseline evaluation"""
    print("📊 Evaluating model...")
    metrics = model.val(data=yaml_path)

    return metrics


# Run validation
val_metrics = simple_evaluate_model(model)


📊 Evaluating model...
Ultralytics 8.3.204  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
YOLO11n summary (fused): 100 layers, 2,582,542 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1391.6932.7 MB/s, size: 66.1 KB)
val: Scanning C:\Users\plebj\Desktop\School\CS5814\rebalanced_dataset\valid\labels.cache... 356 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 356/356  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 14.1it/s 1.6s0.1s
                   all        356        365      0.997      0.997      0.995      0.865
               raccoon        151        162          1          1      0.995        0.9
                rodent        202        203      0.993      0.995      0.995      0.831
Speed: 0.4ms preprocess, 2.0ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to C:\Users\plebj\Desktop\School\CS5814\runs\detect\train2


In [9]:
def evaluate_model(model, yaml_path="data.yaml"):
    """Comprehensive model evaluation"""
    print("📊 Evaluating model...")
    
    # Validation metrics
    metrics = model.val(
        data=yaml_path,
        split="val",
        save_json=True,    # Save results to JSON
        save_hybrid=True,  # Save hybrid version of labels
        conf=0.5,          # Object confidence threshold
        iou=0.6,           # NMS IoU threshold
        plots=True         # Save plots
    )
    
    # Test set evaluation  
    test_metrics = model.val(
        data=yaml_path,
        split="test",
        conf=0.5,
        iou=0.6
    )
    
    return metrics, test_metrics

# Evaluate the trained model
val_metrics, test_metrics = evaluate_model(model)

📊 Evaluating model...
WARNING 'save_hybrid' is deprecated and will be removed in the future.
Ultralytics 8.3.204  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 838.6446.1 MB/s, size: 49.6 KB)
val: Scanning C:\Users\plebj\Desktop\School\CS5814\rebalanced_dataset\valid\labels.cache... 356 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 356/356  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 13.3it/s 1.7s0.1s
                   all        356        365      0.998      0.998      0.995      0.869
               raccoon        151        162          1          1      0.995      0.903
                rodent        202        203      0.995      0.995      0.995      0.835
Speed: 0.2ms preprocess, 1.9ms inference, 0.0ms loss, 0.5ms postprocess per image
Saving C:\Users\plebj\Desktop\School\CS5814\runs\detect\train3\predictions.json...
Resul

In [10]:
def export_model(model, export_formats=["onnx", "tflite"]):
    """Export model to various formats"""
    print("📤 Exporting model...")
    
    exported_paths = {}
    
    for format in export_formats:
        try:
            if format == "tflite":
                # TFLite export for Raspberry Pi
                path = model.export(format=format, int8=True, data="data.yaml")
            else:
                path = model.export(format=format)
            
            exported_paths[format] = path
            print(f"✅ Exported to {format}: {path}")
            
        except Exception as e:
            print(f"❌ Failed to export {format}: {e}")
    
    return exported_paths

# Export the model
exported_paths = export_model(model, export_formats=["onnx"])  # "tflite" if needed

📤 Exporting model...
Ultralytics 8.3.204  Python-3.11.9 torch-2.5.1+cu121 CPU (AMD Ryzen 7 9700X 8-Core Processor)

PyTorch: starting from 'C:\Users\plebj\Desktop\School\CS5814\runs\detect\train\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (5.2 MB)

ONNX: starting export with onnx 1.19.0 opset 19...
ONNX: slimming with onnxslim 0.1.70...
ONNX: export success  1.0s, saved as 'C:\Users\plebj\Desktop\School\CS5814\runs\detect\train\weights\best.onnx' (10.1 MB)

Export complete (1.1s)
Results saved to C:\Users\plebj\Desktop\School\CS5814\runs\detect\train\weights
Predict:         yolo predict task=detect model=C:\Users\plebj\Desktop\School\CS5814\runs\detect\train\weights\best.onnx imgsz=640  
Validate:        yolo val task=detect model=C:\Users\plebj\Desktop\School\CS5814\runs\detect\train\weights\best.onnx imgsz=640 data=data.yaml  
Visualize:       https://netron.app
✅ Exported to onnx: C:\Users\plebj\Desktop\School\CS5814\runs\detect\train\we

In [11]:
def test_inference(model, dataset_path="augmentation_dataset"):
    """Test model inference on sample images"""
    print("🧪 Testing inference...")
    
    test_dir = os.path.join(dataset_path, "test/images")
    if os.path.exists(test_dir):
        test_images = [f for f in os.listdir(test_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        
        if test_images:
            # Test on first 3 images
            for i, img_name in enumerate(test_images[:3]):
                img_path = os.path.join(test_dir, img_name)
                print(f"Testing: {img_name}")
                
                results = model(img_path, conf=0.60)  # Confidence threshold
                
                # Print detection info
                for r in results:
                    if len(r.boxes) > 0:
                        print(f"  Detected {len(r.boxes)} pests")
                    else:
                        print("  No pests detected")
            
            print("✅ Inference test completed!")
        else:
            print("⚠️  No test images found")
    else:
        print("⚠️  Test directory not found")

# Test inference
test_inference(model, DATASET_PATH)

🧪 Testing inference...
Testing: 00b8f682-b70d-4dc8-a813-e6c2e8baa232.jpg

image 1/1 c:\Users\plebj\Desktop\School\CS5814\rebalanced_dataset\test\images\00b8f682-b70d-4dc8-a813-e6c2e8baa232.jpg: 640x640 1 raccoon, 4.5ms
Speed: 1.6ms preprocess, 4.5ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 640)
  Detected 1 pests
Testing: 00cc44cb-d257-4ab4-a082-b77e5625b42a.jpg

image 1/1 c:\Users\plebj\Desktop\School\CS5814\rebalanced_dataset\test\images\00cc44cb-d257-4ab4-a082-b77e5625b42a.jpg: 640x640 1 rodent, 5.5ms
Speed: 1.2ms preprocess, 5.5ms inference, 0.7ms postprocess per image at shape (1, 3, 640, 640)
  Detected 1 pests
Testing: 03012f1f-9ae3-434e-aa56-5043401b186b.jpg

image 1/1 c:\Users\plebj\Desktop\School\CS5814\rebalanced_dataset\test\images\03012f1f-9ae3-434e-aa56-5043401b186b.jpg: 640x640 1 raccoon, 4.5ms
Speed: 1.6ms preprocess, 4.5ms inference, 1.5ms postprocess per image at shape (1, 3, 640, 640)
  Detected 1 pests
✅ Inference test completed!


In [12]:
print("\n🎉 Training completed successfully!")
print(f"📁 Model saved in: {os.path.join('runs', 'detect', 'train')}")
import json

# --- Print final metrics ---
if hasattr(results, 'results_dict'):
    print(f"📊 Final mAP@0.5: {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")

if test_metrics is not None:
    # Prefer results_dict for broader version compatibility
    if hasattr(test_metrics, 'results_dict'):
        precision = test_metrics.results_dict.get('metrics/precision(B)', None)
        recall = test_metrics.results_dict.get('metrics/recall(B)', None)
        map50 = test_metrics.results_dict.get('metrics/mAP50(B)', None)
        map5095 = test_metrics.results_dict.get('metrics/mAP50-95(B)', None)

        print(f"📊 Test Precision: {precision:.4f}")
        print(f"📊 Test Recall: {recall:.4f}")
        print(f"📊 Test mAP@0.5: {map50:.4f}")
        print(f"📊 Test mAP@0.5:0.95: {map5095:.4f}")
    else:
        print("⚠️ test_metrics does not contain results_dict.")
else:
    print("📊 Test metrics not available (test_metrics is None)")





🎉 Training completed successfully!
📁 Model saved in: runs\detect\train
📊 Final mAP@0.5: 0.9948522167487686
📊 Test Precision: 0.9840
📊 Test Recall: 0.9947
📊 Test mAP@0.5: 0.9942
📊 Test mAP@0.5:0.95: 0.8789


In [ ]:
# ============================================================
# Enhanced Evaluation Suite (Original + Robustness Testing)
# ============================================================

import os
from pathlib import Path
import shutil
from ultralytics import YOLO
import numpy as np
import matplotlib.pyplot as plt
import random
import albumentations as A
import cv2
from tqdm import tqdm

# --- Config ---
MODEL_PATH = Path("runs/detect/train/weights/best.pt")
BASE_DATASET = Path("rebalanced_dataset")  # dataset used during training

# --- Load model ---
print(f"📦 Loading YOLO model from {MODEL_PATH} ...")
model = YOLO(MODEL_PATH)
print("✅ Model loaded successfully!")

# ============================================================
# Test A: Evaluate on Original Test Split
# ============================================================
print("\n🧪 [Test A] Evaluating on Original Test Split...")
results_orig = model.val(
    data="data.yaml",
    split="test",
    conf=0.6,
    save_json=True,
    plots=True
)
print("✅ Original test split evaluation complete!")

# ============================================================
# Test B: Robust Environmental Testing (FIXED)
# ============================================================
print("\n🧪 [Test B] Testing Environmental Robustness...")

def create_proper_environmental_dataset():
    """Create a proper YOLO dataset with environmental augmentations"""
    env_dataset = Path("environmental_test_dataset")
    
    # Create proper YOLO directory structure
    (env_dataset / "images").mkdir(parents=True, exist_ok=True)
    (env_dataset / "labels").mkdir(parents=True, exist_ok=True)
    
    # Conservative environmental augmentations
    env_augmentations = A.Compose([
        # Lighting variations (day/night)
        A.RandomBrightnessContrast(brightness_limit=(-0.3, 0.2), contrast_limit=0.2, p=0.6),
        A.RandomGamma(gamma_limit=(70, 130), p=0.4),
        
        # Weather/Environmental noise
        A.MotionBlur(blur_limit=5, p=0.3),
        A.GaussNoise(var_limit=(5, 25), p=0.3),
        
        # Mild weather effects
        A.RandomFog(fog_coef_lower=0.05, fog_coef_upper=0.2, alpha_coef=0.04, p=0.2),
    ])
    
    # Get source images and labels
    test_images_dir = BASE_DATASET / "test/images"
    test_labels_dir = BASE_DATASET / "test/labels"
    
    test_images = list(test_images_dir.glob("*.*"))
    
    if not test_images:
        print("⚠️  No test images found for environmental testing")
        return None
    
    print(f"🔄 Creating environmental test set with {len(test_images)} images...")
    
    for img_path in tqdm(test_images, desc="Applying environmental augmentations"):
        if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
            img = cv2.imread(str(img_path))
            if img is not None:
                # Apply environmental augmentations
                aug_img = env_augmentations(image=img)["image"]
                
                # Save augmented image
                cv2.imwrite(str(env_dataset / "images" / img_path.name), aug_img)
                
                # Copy corresponding label file
                label_path = test_labels_dir / f"{img_path.stem}.txt"
                if label_path.exists():
                    shutil.copy2(label_path, env_dataset / "labels" / f"{img_path.stem}.txt")
    
    # Create a simple data.yaml for the environmental dataset
    env_yaml_content = f"""
path: {env_dataset.absolute()}
train: ../train/images
val: images
test: images

nc: 2
names: {{
  0: 'raccoon',
  1: 'rodent'
}}
"""
    with open(env_dataset / "data.yaml", "w") as f:
        f.write(env_yaml_content)
    
    return env_dataset

# Create and test environmental dataset
env_dataset = create_proper_environmental_dataset()
results_env = None

if env_dataset and (env_dataset / "images").exists():
    print("🌧️  Testing model robustness to environmental conditions...")
    try:
        results_env = model.val(
            data=str(env_dataset / "data.yaml"),
            conf=0.5,
            save=True,
            project="environmental_test_results"
        )
        print("✅ Environmental robustness test complete!")
    except Exception as e:
        print(f"⚠️  Environmental validation failed: {e}")
        print("🔄 Using fallback prediction method...")
        
        # Fallback: Use predict on environmental images
        env_images = list((env_dataset / "images").glob("*.*"))
        if env_images:
            results_env_fallback = model.predict(
                source=str(env_dataset / "images"),
                conf=0.5,
                save=True,
                project="environmental_test_fallback"
            )
            print("✅ Environmental test completed with prediction fallback")
else:
    print("⚠️  Could not create environmental test dataset")

# ============================================================
# Test C: Simple Robustness Check (Reliable)
# ============================================================
print("\n🧪 [Test C] Simple Robustness Check...")

def simple_robustness_check(model, num_test_images=10):
    """Quick reliability test with basic augmentations"""
    test_images_dir = BASE_DATASET / "test/images"
    test_images = list(test_images_dir.glob("*.*"))[:num_test_images]
    
    if not test_images:
        print("⚠️  No test images found for robustness check")
        return None
    
    # Simple augmentations that simulate real-world variations
    simple_aug = A.Compose([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.1, p=1.0),
        A.GaussNoise(var_limit=10.0, p=0.5),
    ])
    
    detection_results = []
    confidence_scores = []
    
    for img_path in test_images:
        original_img = cv2.imread(str(img_path))
        if original_img is not None:
            # Test original image
            original_results = model(original_img, conf=0.5, verbose=False)
            original_detections = len(original_results[0].boxes)
            
            # Test augmented image
            aug_img = simple_aug(image=original_img)["image"]
            aug_results = model(aug_img, conf=0.5, verbose=False)
            aug_detections = len(aug_results[0].boxes)
            
            # Collect confidence scores
            if aug_results[0].boxes:
                confidence_scores.extend(aug_results[0].boxes.conf.cpu().numpy())
            
            detection_results.append({
                'image': img_path.name,
                'original_detections': original_detections,
                'augmented_detections': aug_detections,
                'difference': aug_detections - original_detections
            })
    
    # Analyze results
    if detection_results:
        differences = [r['difference'] for r in detection_results]
        avg_difference = np.mean(differences)
        std_difference = np.std(differences)
        
        print(f"📊 Robustness Check Results ({len(detection_results)} images):")
        print(f"   Average detection change: {avg_difference:+.2f}")
        print(f"   Consistency (std dev): {std_difference:.2f}")
        
        if abs(avg_difference) < 0.5 and std_difference < 1.0:
            print("   ✅ Excellent robustness - stable under variations")
        elif abs(avg_difference) < 1.0 and std_difference < 1.5:
            print("   ✅ Good robustness - minor variations")
        else:
            print("   ⚠️  Sensitivity detected - performance varies with conditions")
        
        if confidence_scores:
            avg_confidence = np.mean(confidence_scores)
            print(f"   Average confidence: {avg_confidence:.3f}")
    
    return detection_results

# Run simple robustness check
robustness_results = simple_robustness_check(model, num_test_images=8)
print("✅ Simple robustness check complete!")

# ============================================================
# Test D: Confidence Distribution Analysis
# ============================================================
print("\n🧪 [Test D] Confidence Distribution Analysis...")

def analyze_confidence_distribution(model, num_samples=20):
    """Analyze confidence scores across different conditions"""
    test_images_dir = BASE_DATASET / "test/images"
    test_images = list(test_images_dir.glob("*.*"))[:num_samples]
    
    if not test_images:
        return None
    
    confidence_data = {
        'high_light': [],
        'low_light': [],
        'noisy': []
    }
    
    # Create different condition augmentations
    high_light_aug = A.RandomBrightnessContrast(brightness_limit=(0.1, 0.3), p=1.0)
    low_light_aug = A.RandomBrightnessContrast(brightness_limit=(-0.4, -0.1), p=1.0)
    noisy_aug = A.Compose([
        A.GaussNoise(var_limit=20.0, p=1.0),
        A.MotionBlur(blur_limit=3, p=0.5)
    ])
    
    for img_path in test_images:
        img = cv2.imread(str(img_path))
        if img is not None:
            # Test different conditions
            conditions = [
                ('high_light', high_light_aug(image=img)["image"]),
                ('low_light', low_light_aug(image=img)["image"]),
                ('noisy', noisy_aug(image=img)["image"])
            ]
            
            for condition_name, condition_img in conditions:
                results = model(condition_img, conf=0.3, verbose=False)  # Lower threshold to see more detections
                if results[0].boxes:
                    confidence_data[condition_name].extend(results[0].boxes.conf.cpu().numpy())
    
    # Print confidence analysis
    print("📊 Confidence Distribution by Condition:")
    for condition, scores in confidence_data.items():
        if scores:
            avg_conf = np.mean(scores)
            std_conf = np.std(scores)
            count = len(scores)
            print(f"   {condition.replace('_', ' ').title():12}: {avg_conf:.3f} ± {std_conf:.3f} ({count} detections)")
        else:
            print(f"   {condition.replace('_', ' ').title():12}: No detections")
    
    return confidence_data

confidence_analysis = analyze_confidence_distribution(model)
print("✅ Confidence analysis complete!")

# ============================================================
# Enhanced Combined Summary (FIXED)
# ============================================================
print("\n" + "="*60)
print("🎯 COMPREHENSIVE MODEL EVALUATION SUMMARY")
print("="*60)

# Helper function to safely extract and format metrics
def safe_get_metric(metrics_obj, attr_path, default="N/A"):
    """Safely extract metric values and handle NumPy arrays"""
    try:
        # Navigate through attribute path (e.g., "box.map50")
        parts = attr_path.split('.')
        value = metrics_obj
        for part in parts:
            value = getattr(value, part)
        
        # Handle NumPy arrays and scalars
        if hasattr(value, 'item'):  # NumPy array/scalar
            return float(value.item())
        elif isinstance(value, (int, float)):
            return float(value)
        else:
            return default
    except (AttributeError, ValueError, TypeError):
        return default

# Original Test Results
try:
    print(f"\n🧪 ORIGINAL TEST SET (Clean Conditions):")
    map50 = safe_get_metric(results_orig, 'box.map50', 0)
    map5095 = safe_get_metric(results_orig, 'box.map', 0)
    precision = safe_get_metric(results_orig, 'box.p', 0)
    recall = safe_get_metric(results_orig, 'box.r', 0)
    
    print(f"   mAP@50:      {map50:.3f}")
    print(f"   mAP@50-95:   {map5095:.3f}")
    print(f"   Precision:   {precision:.3f}")
    print(f"   Recall:      {recall:.3f}")
    
    # Print class-wise metrics if available
    if hasattr(results_orig, 'results_dict'):
        raccoon_map = results_orig.results_dict.get('raccoon/mAP50(B)', 'N/A')
        rodent_map = results_orig.results_dict.get('rodent/mAP50(B)', 'N/A')
        print(f"   Raccoon mAP: {raccoon_map}")
        print(f"   Rodent mAP:  {rodent_map}")
        
except Exception as e:
    print(f"⚠️  Could not access original test metrics: {e}")

# Environmental Test Results
if results_env is not None:
    try:
        env_map50 = safe_get_metric(results_env, 'box.map50', 0)
        env_precision = safe_get_metric(results_env, 'box.p', 0)
        env_recall = safe_get_metric(results_env, 'box.r', 0)
        
        print(f"\n🌧️  ENVIRONMENTAL TEST SET:")
        print(f"   mAP@50:    {env_map50:.3f}")
        print(f"   Precision: {env_precision:.3f}")
        print(f"   Recall:    {env_recall:.3f}")
        
        # Robustness analysis
        if map50 > 0:  # Only calculate if we have valid original metrics
            env_drop = map50 - env_map50
            robustness_pct = (env_map50 / map50) * 100 if map50 > 0 else 0
            
            print(f"\n🛡️  ROBUSTNESS ANALYSIS:")
            print(f"   Performance drop:   {env_drop:.3f}")
            print(f"   Robustness:        {robustness_pct:.1f}% of original")
            
            if env_drop < 0.1:
                print("   ✅ EXCELLENT - Minimal performance loss in tough conditions")
            elif env_drop < 0.2:
                print("   ✅ GOOD - Reasonable performance in varied conditions")
            else:
                print("   ⚠️  MODERATE - Significant drop, consider environmental training")
                
    except Exception as e:
        print(f"⚠️  Could not access environmental test metrics: {e}")

# Robustness Check Results
if robustness_results:
    print(f"\n🔍 ROBUSTNESS CHECK DETAILS:")
    differences = [r['difference'] for r in robustness_results]
    avg_difference = np.mean(differences)
    std_difference = np.std(differences)
    
    print(f"   Detection stability: {avg_difference:+.2f} ± {std_difference:.2f}")
    
    if abs(avg_difference) < 0.5 and std_difference < 1.0:
        print("   ✅ Excellent consistency across conditions")
    elif abs(avg_difference) < 1.0 and std_difference < 1.5:
        print("   ✅ Good consistency")
    else:
        print("   ⚠️  Variable performance across conditions")

# Confidence Analysis Results
if confidence_analysis:
    print(f"\n📊 CONFIDENCE ANALYSIS:")
    for condition, scores in confidence_analysis.items():
        if scores:
            avg_conf = np.mean(scores)
            std_conf = np.std(scores)
            count = len(scores)
            condition_name = condition.replace('_', ' ').title()
            print(f"   {condition_name:12}: {avg_conf:.3f} ± {std_conf:.3f} ({count} detections)")

# Final Recommendations
print(f"\n💡 RECOMMENDATIONS:")
if map50 > 0.95:
    print("   🎉 BASE MODEL: Excellent performance - ready for deployment")
elif map50 > 0.85:
    print("   ✅ BASE MODEL: Very good performance - production ready")
elif map50 > 0.75:
    print("   🔧 BASE MODEL: Good performance - consider fine-tuning")
else:
    print("   🛠️  BASE MODEL: Needs improvement - review training data")

if env_map50 > 0 and (map50 - env_map50) > 0.2:
    print("   🌦️  ENVIRONMENT: Significant performance drop - add weather augmentations to training")
elif env_map50 > 0 and (map50 - env_map50) > 0.1:
    print("   🌦️  ENVIRONMENT: Moderate drop - monitor in production")
else:
    print("   🌦️  ENVIRONMENT: Good robustness - handles variations well")

print(f"\n📁 Results saved in:")
print(f"   - runs/detect/val/ (original test)")
if env_dataset and env_dataset.exists():
    print(f"   - environmental_test_results/ (environmental test)")

# Cleanup
for temp_dir in ["environmental_test_dataset", "environmental_aug_test"]:
    temp_path = Path(temp_dir)
    if temp_path.exists():
        shutil.rmtree(temp_path)
        print(f"🧹 Cleaned up {temp_dir}")

print("\n✅ All evaluation tests completed!")

📦 Loading YOLO model from runs\detect\train\weights\best.pt ...
✅ Model loaded successfully!

🧪 [Test A] Evaluating on Original Test Split...
Ultralytics 8.3.204  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
YOLO11n summary (fused): 100 layers, 2,582,542 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 916.5531.4 MB/s, size: 68.8 KB)
val: Scanning C:\Users\plebj\Desktop\School\CS5814\rebalanced_dataset\test\labels.cache... 178 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 178/178 178.0Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 11.2it/s 1.1s0.1s
                   all        178        181      0.984      0.995      0.994      0.879
               raccoon         73         78      0.987          1      0.994      0.899
                rodent        102        103      0.981       0.99      0.994      0.859
Speed: 0.4ms preprocess, 2.6m

Applying environmental augmentations: 100%|██████████| 178/178 [00:06<00:00, 25.69it/s]

🌧️  Testing model robustness to environmental conditions...
Ultralytics 8.3.204  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
val: Fast image access  (ping: 0.00.0 ms, read: 14.36.9 MB/s, size: 77.3 KB)


val: Scanning C:\Users\plebj\Desktop\School\CS5814\environmental_test_dataset\labels... 178 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 178/178 1.5Kit/s 0.1s<0.0s
val: New cache created: C:\Users\plebj\Desktop\School\CS5814\environmental_test_dataset\labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 12/12 12.7it/s 0.9s0.1s
                   all        178        181      0.938      0.699      0.833      0.684
               raccoon         73         78      0.947      0.718      0.848      0.713
                rodent        102        103      0.928       0.68      0.818      0.655
Speed: 0.3ms preprocess, 2.5ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to C:\Users\plebj\Desktop\School\CS5814\environmental_test_results\val
✅ Environmental robustness test complete!

🧪 [Test C] Simple Robustness Check...
📊 Robustness Check Results (8 images):
   Average detection change: -0.88
   Cons

TypeError: unsupported format string passed to numpy.ndarray.__format__